In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from collections import Counter
import re

# Set device to GPU if available otherwise use CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [2]:
#Load data
train_df = pd.read_csv('train.csv', encoding='ISO-8859-1').dropna(subset=['text', 'sentiment'])
test_df = pd.read_csv('test.csv', encoding='ISO-8859-1').dropna(subset=['text', 'sentiment'])

# Encode labels
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(train_df['sentiment'])
y_test = label_encoder.transform(test_df['sentiment'])


In [3]:
#Text Tokenization and Vocab
VOCAB_SIZE = 10000
MAX_LENGTH = 50

def tokenize(text):
    text = re.sub(r"[^a-zA-Z\s]", "", str(text).lower())
    return text.split()
    
# Count all words in training set
all_words = []
for text in train_df['text']:
    all_words.extend(tokenize(text))

# Get the most common 10,000 words
word_counts = Counter(all_words)
common_words = [word for word, count in word_counts.most_common(VOCAB_SIZE - 2)]

# Create mapping (0 for padding, 1 for unknown words)
word2idx = {word: i+2 for i, word in enumerate(common_words)}
word2idx['<PAD>'] = 0
word2idx['<UNK>'] = 1

def encode_and_pad(text):
    tokens = tokenize(text)
    # Convert to indices
    encoded = [word2idx.get(word, word2idx['<UNK>']) for word in tokens]
    # Truncate or Pad to MAX_LENGTH
    if len(encoded) < MAX_LENGTH:
        encoded += [word2idx['<PAD>']] * (MAX_LENGTH - len(encoded))
    else:
        encoded = encoded[:MAX_LENGTH]
    return encoded

X_train_encoded = [encode_and_pad(text) for text in train_df['text']]
X_test_encoded = [encode_and_pad(text) for text in test_df['text']]

In [4]:
#PyTorch Dataset and DataLoader
class TweetDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = torch.tensor(texts, dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)
        
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.texts[idx], self.labels[idx]

train_dataset = TweetDataset(X_train_encoded, y_train)
test_dataset = TweetDataset(X_test_encoded, y_test)

# DataLoader (feed data to the model in batches)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [5]:
#PyTorch Neural Network Architecture
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim, n_layers, bidirectional, dropout):
        super().__init__()
        # Embedding Layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        # Bidirectional LSTM Layer
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=n_layers, 
                            bidirectional=bidirectional, batch_first=True)
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        # Fully Connected Output Layer
        self.fc = nn.Linear(hidden_dim * 2 if bidirectional else hidden_dim, output_dim)
        
    def forward(self, text):
        # text shape: [batch size, sequence length]
        embedded = self.dropout(self.embedding(text))
        
        # lstm_out shape: [batch, seq len, hidden_dim * 2]
        # hidden shape: [num_layers * 2, batch, hidden_dim]
        lstm_out, (hidden, cell) = self.lstm(embedded)
        
        # We concatenate the final hidden states from both directions (forward and backward)
        hidden = self.dropout(torch.cat((hidden[-2,:,:], hidden[-1,:,:]), dim=1))
        
        # Pass to the final output layer
        return self.fc(hidden)

model = SentimentLSTM(
    vocab_size=VOCAB_SIZE, 
    embed_dim=256, 
    hidden_dim=128, 
    output_dim=3, # Negative, Neutral, Positive
    n_layers=1, 
    bidirectional=True, 
    dropout=0.05
).to(device)

# Define Loss function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)



In [6]:
#Training Loop
EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    epoch_acc = 0
    
    for batch_texts, batch_labels in train_loader:
        # Move data to GPU if available
        batch_texts, batch_labels = batch_texts.to(device), batch_labels.to(device)
        
        # 1. Zero the gradients
        optimizer.zero_grad()
        
        # 2. Forward pass
        predictions = model(batch_texts)
        
        # 3. Calculate Loss and Accuracy
        loss = criterion(predictions, batch_labels)
        
        # 4. Backward pass (calculate gradients)
        loss.backward()
        
        # 5. Update weights
        optimizer.step()
        
        epoch_loss += loss.item()
        
    print(f"Epoch: {epoch+1:02} | Train Loss: {epoch_loss/len(train_loader):.3f}")

Epoch: 01 | Train Loss: 0.894
Epoch: 02 | Train Loss: 0.696
Epoch: 03 | Train Loss: 0.578
Epoch: 04 | Train Loss: 0.466
Epoch: 05 | Train Loss: 0.363
Epoch: 06 | Train Loss: 0.275
Epoch: 07 | Train Loss: 0.201
Epoch: 08 | Train Loss: 0.151
Epoch: 09 | Train Loss: 0.112
Epoch: 10 | Train Loss: 0.082
Epoch: 11 | Train Loss: 0.070
Epoch: 12 | Train Loss: 0.066
Epoch: 13 | Train Loss: 0.048
Epoch: 14 | Train Loss: 0.046
Epoch: 15 | Train Loss: 0.042
Epoch: 16 | Train Loss: 0.041
Epoch: 17 | Train Loss: 0.035
Epoch: 18 | Train Loss: 0.027
Epoch: 19 | Train Loss: 0.028
Epoch: 20 | Train Loss: 0.037
Epoch: 21 | Train Loss: 0.028
Epoch: 22 | Train Loss: 0.024
Epoch: 23 | Train Loss: 0.022
Epoch: 24 | Train Loss: 0.021
Epoch: 25 | Train Loss: 0.025


In [7]:
#Model Evaluation
print("\nEvaluating on unseen Test Set...")
model.eval()
all_preds = []
all_labels = []

# Disable gradient calculation for testing to save memory and speed up
with torch.no_grad():
    for batch_texts, batch_labels in test_loader:
        batch_texts = batch_texts.to(device)
        
        predictions = model(batch_texts)
        # Get the index of the highest probability
        max_preds = predictions.argmax(dim=1, keepdim=True).squeeze(1)
        
        all_preds.extend(max_preds.cpu().numpy())
        all_labels.extend(batch_labels.numpy())

# Print the final report
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))


Evaluating on unseen Test Set...

Classification Report:
              precision    recall  f1-score   support

    negative       0.68      0.66      0.67      1001
     neutral       0.63      0.65      0.64      1430
    positive       0.74      0.74      0.74      1103

    accuracy                           0.68      3534
   macro avg       0.68      0.68      0.68      3534
weighted avg       0.68      0.68      0.68      3534

